# PyDI Data Integration Workflow: Companies

This notebook demonstrates comprehensive data integration using PyDI. We'll work with companies datasets to showcase the data integration pipeline from entity matching to data fusion.

## Table of Contents
  - [Datasets](#datasets)
- [Part 1: Data Loading and Profiling](#part-1-data-loading-and-profiling)
- [Part 2: Entity Matching](#part-2-entity-matching)
  - [Step 1: Blocking](#step-1-blocking)
  - [Step 2: Blocking Evaluation](#step-2-evaluate-blocking-against-ground-truth)
  - [Step 3: Entity Matching with Comparators](#step-3-entity-matching-with-comparators)
  - [Step 4: Entity Matching Evaluation](#step-4-evaluate-matching-against-ground-truth)
- [Part 3: Data Fusion](#part-3-data-fusion)
  - [Step 1: Define Fusion Strategy](#step-1-define-fusion-strategy)
  - [Step 2: Run Fusion](#step-2-run-fusion)
  - [Step 3: Data Fusion Evaluation](#step-3-evaluate-data-fusion)

from pathlib import Path

# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input" / "companies"
OUTPUT_DIR = NOTEBOOK_DIR / "output" / "companies"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [1]:
from utils import get_repo_root

ROOT = get_repo_root()
INPUT_DIR = ROOT / "usecases" / "input" / "companies"
OUTPUT_DIR = ROOT / "usecases" / "output" / "companies"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ModuleNotFoundError: No module named 'utils'

## Part 1: Data Loading and Profiling

In [ ]:
from PyDI.io import load_xml

# Load DBpedia dataset
dbpedia = load_xml(
    INPUT_DIR / "data" / "dbpedia.xml",
    name="dbpedia",
    nested_handling="aggregate"
)

# Load Forbes dataset
forbes = load_xml(
    INPUT_DIR / "data" / "forbes.xml",
    name="forbes",
    nested_handling="aggregate"
)

# Load Last.fm dataset
fullcontact = load_xml(
    INPUT_DIR / "data" / "fullcontact.xml",
    name="fullcontact",
    nested_handling="aggregate"
)

# Display basic information
datasets = [dbpedia, forbes, fullcontact]
names = ["DBpedia", "Forbes", "FullContact"]

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Total records across all datasets: 14,017


In [ ]:
from PyDI.utils import DataProfiler

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

dbpedia:
  Rows: 10,086
  Columns: 9
  Total nulls: 30,516
  Null percentage: 33.6%
  Null counts per column:
    city: 700 (6.9%)
    industry: 3,208 (31.8%)
    keypeople_name: 9,158 (90.8%)
    assets: 9,426 (93.5%)
    revenue: 8,024 (79.6%)

forbes:
  Rows: 2,000
  Columns: 7
  Total nulls: 114
  Null percentage: 0.8%
  Null counts per column:
    country: 71 (3.5%)
    industry: 43 (2.1%)

fullcontact:
  Rows: 1,931
  Columns: 6
  Total nulls: 3,586
  Null percentage: 31.0%
  Null counts per column:
    country: 508 (26.3%)
    city: 464 (24.0%)
    keypeople_name: 1,739 (90.1%)
    founded: 875 (45.3%)



{'rows': 1931,
 'columns': 6,
 'nulls_total': 3586,
 'nulls_per_column': {'id': 0,
  'name': 0,
  'country': 508,
  'city': 464,
  'keypeople_name': 1739,
  'founded': 875},
 'dtypes': {'id': 'object',
  'name': 'object',
  'country': 'object',
  'city': 'object',
  'keypeople_name': 'object',
  'founded': 'object'}}

### Attribute Coverage Analysis

In [ ]:
coverage = profiler.analyze_coverage(
    datasets=datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("📊 Attribute coverage across datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

[INFO ] PyDI.fusion.analysis - Analyzed 10 attributes across 3 datasets


📊 Attribute coverage across datasets:


,attribute,dbpedia_count,dbpedia_pct,dbpedia_coverage,dbpedia_samples,forbes_count,forbes_pct,forbes_coverage,forbes_samples,fullcontact_count,fullcontact_pct,fullcontact_coverage,fullcontact_samples,avg_coverage,max_coverage,datasets_with_attribute
0,assets,660/10086,6.5%,0.065437,"['8', '240560000000', '607100000']",2000/2000,100.0%,1.0000,"['3124900000000', '2449500000000', '2405400000...",0/0,0%,0.000000,N/A,0.355146,1.000000,2
1,city,9386/10086,93.1%,0.930597,"['Castelnaudary', 'Lisbon', 'Mexico City']",0/0,0%,0.0000,N/A,1467/1931,76.0%,0.759710,"['Brooklyn', 'Toronto', 'Waterloo']",0.563436,0.930597,2
2,country,10086/10086,100.0%,1.000000,"['France', 'Portugal', 'Mexico']",1929/2000,96.5%,0.9645,"['China', 'China', 'China']",1423/1931,73.7%,0.736924,"['United States', 'Canada', 'United States']",0.900475,1.000000,3
3,founded,10086/10086,100.0%,1.000000,"['1970-01-01', '1993-01-01', '2002-01-01']",0/0,0%,0.0000,N/A,1056/1931,54.7%,0.546867,"['1908-01-01', '1957-01-01', '1871-01-01']",0.515622,1.000000,2
4,id,10086/10086,100.0%,1.000000,['http://dbpedia.org/resource/%C3%80_la_Table_...,2000/2000,100.0%,1.0000,"['http://www.forbes.com/companies/icbc/', 'htt...",1931/1931,100.0%,1.000000,"['fullcontact_1', 'fullcontact_2', 'fullcontac...",1.000000,1.000000,3
5,industry,6878/10086,68.2%,0.681935,"['Meat', 'Animation', 'Communication']",1957/2000,97.9%,0.9785,"['Major Banks', 'Regional Banks', 'Regional Ba...",0/0,0%,0.000000,N/A,0.553478,0.978500,2
6,keypeople_name,928/10086,9.2%,0.092009,"['Çalık Holding', 'Ahmet Çalık', 'Marcel Paul']",0/0,0%,0.0000,N/A,192/1931,9.9%,0.099430,"['Raphael Bemporad', 'John Pitcairn', 'Douglas...",0.063813,0.099430,2
7,name,10086/10086,100.0%,1.000000,"['À la Table de Spanghero', '�?guas de Portuga...",2000/2000,100.0%,1.0000,"['ICBC', 'China Construction Bank', 'Agricultu...",1931/1931,100.0%,1.000000,"['BBMG', 'CIT Group Inc (DEL)', 'City & Nation...",1.000000,1.000000,3
8,revenue,2062/10086,20.4%,0.204442,"['2.8', '65170000000', '358500000']",2000/2000,100.0%,1.0000,"['148700000000', '121300000000', '136400000000']",0/0,0%,0.000000,N/A,0.401481,1.000000,2
9,website,0/0,0%,0.000000,N/A,2000/2000,100.0%,1.0000,"['http://www.forbes.com/companies/icbc/', 'htt...",0/0,0%,0.000000,N/A,0.333333,1.000000,1



🔗 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['assets', 'city', 'country', 'founded', 'id', 'industry', 'keypeople_name', 'name', 'revenue']


## Part 2: Entity Matching

### Step 1: Blocking

In [ ]:
# Set up logging
import logging

import os
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

In [ ]:
from PyDI.entitymatching import StandardBlocker, SortedNeighbourhoodBlocker, TokenBlocker, EmbeddingBlocker
import re

# Standard Blocking - First meaningful token in the name
def generate_blocking_keys_tokens(company_name: str):
    tokens = re.split(r'[^a-z]', company_name.lower())
    first_token = [token for token in tokens if len(token) > 1]
    if first_token:
        return first_token[0]
    else:
        return company_name # Return full string if no valid token found
    

# Add first-token column to the original dataframes used for blocking
dbpedia['name_first_token'] = dbpedia['name'].apply(generate_blocking_keys_tokens)
forbes['name_first_token'] = forbes['name'].apply(generate_blocking_keys_tokens)
fullcontact['name_first_token'] = fullcontact['name'].apply(generate_blocking_keys_tokens)

standard_blocker_f2d = StandardBlocker(
    forbes, dbpedia,
    on=['name_first_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
standard_candidates_f2d = standard_blocker_f2d.materialize()

# Forbes vs FullContact
standard_blocker_f2fc = StandardBlocker(
    forbes, fullcontact,
    on=['name_first_token'],
    batch_size=1000,
    output_dir=OUTPUT_DIR / "blocking-evaluation",
    id_column='id'
)
standard_candidates_f2fc = standard_blocker_f2fc.materialize()

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1647 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 7922 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 658 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1647 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 1750 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 776 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/luca/PycharmProjects/PyDI/usecases/output

### Step 2: Evaluate Blocking Against Ground Truth

In [ ]:
import pandas as pd
from PyDI.io import load_csv
from PyDI.entitymatching import EntityMatchingEvaluator
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_dbpedia_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_f2d,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root -   Pair Completeness: 0.952
[INFO ] root -   Pair Quality:      0.013
[INFO ] root -   Reduction Ratio:   0.999630
[INFO ] root -   True Matches Found: 99/104
[INFO ] root -   Batches Processed:  8
[INFO ] root - Blocking evaluation complete!


{'pair_completeness': 0.9519230769230769,
 'pair_quality': 0.01326544285140024,
 'reduction_ratio': 0.9996300317271466,
 'total_candidates': 7463,
 'total_possible_pairs': 20172000,
 'true_positives_found': 99,
 'total_true_pairs': 104,
 'batches_processed': 8,
 'evaluation_timestamp': '2025-12-10T18:12:57.119412',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/blocking_detailed_results.csv']}

In [ ]:
# Load test set with proper column names
test_gt = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_fullcontact_val.csv",
    name="test_set", header=None, names=['id1', 'id2', 'label'], add_index=False
)

# Use EntityMatchingEvaluator.evaluate_blocking_batched on Standard Blocking
results = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_f2fc,
    test_pairs=test_gt,
    out_dir=OUTPUT_DIR / "blocking-evaluation"
)

display(results)

[INFO ] root - Starting batched blocking evaluation...
[INFO ] root -   Pair Completeness: 0.918
[INFO ] root -   Pair Quality:      0.136
[INFO ] root -   Reduction Ratio:   0.999595
[INFO ] root -   True Matches Found: 212/231
[INFO ] root -   Batches Processed:  2
[INFO ] root - Blocking evaluation complete!


{'pair_completeness': 0.9177489177489178,
 'pair_quality': 0.13563659628918745,
 'reduction_ratio': 0.9995952874158467,
 'total_candidates': 1563,
 'total_possible_pairs': 3862000,
 'true_positives_found': 212,
 'total_true_pairs': 231,
 'batches_processed': 2,
 'evaluation_timestamp': '2025-12-10T18:12:57.663302',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/blocking_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/companies/blocking-evaluation/blocking_detailed_results.csv']}

### Step 3: Entity Matching with Comparators

In [ ]:
from PyDI.entitymatching import StringComparator
import re

# ignore case and punctuation
def normalize_text(s: str) -> str: 
    if s is None:
        return ""
    return re.sub(r"[^\w\s]|_", "", s).lower()

comparators_f2d = [
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='name', 
        similarity_function='levenshtein',
        preprocess=normalize_text
    ),
    StringComparator(
        column='country',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='industry',
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
]

comparators_f2fc = [
    StringComparator(
        column='name', 
        similarity_function='jaccard',
        preprocess=normalize_text
    ),
    StringComparator(
        column='country',
        similarity_function='jaccard',
        preprocess=normalize_text
    )
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

In [ ]:
from PyDI.entitymatching import RuleBasedMatcher

# Initialize Rule-Based Matcher
matcher = RuleBasedMatcher()

correspondences_f2d = matcher.match(
    df_left=forbes,
    df_right=dbpedia, 
    candidates=standard_blocker_f2d,
    comparators=comparators_f2d,
    weights=[1.0, 1.0, 1.0, 0.3],
    threshold=0.2,
    id_column='id'
)

correspondences_f2fc = matcher.match(
    df_left=forbes,
    df_right=fullcontact, 
    candidates=standard_blocker_f2fc,
    comparators=comparators_f2fc,
    weights=[1.0, 1.0],
    threshold=0.1,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 2000 x 10086 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 2000 x 10086 elements after 0:00:0.015; 7463 blocked pairs (reduction ratio: 0.9996300317271466)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:2.433; found 5237 correspondences.
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 2000 x 1931 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 2000 x 1931 elements after 0:00:0.003; 1563 blocked pairs (reduction ratio: 0.9995952874158467)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:0.351; found 1454 correspondences.


### Step 4: Evaluate Matching Against Ground Truth

In [ ]:
gt_val = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_dbpedia_val.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2d,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

display(eval_results)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  99
[INFO ] root -   True Negatives:  87
[INFO ] root -   False Positives: 28
[INFO ] root -   False Negatives: 5
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.849
[INFO ] root -   Precision: 0.780
[INFO ] root -   Recall:    0.952
[INFO ] root -   F1-Score:  0.857


{'precision': 0.7795275590551181,
 'recall': 0.9519230769230769,
 'f1': 0.857142857142857,
 'accuracy': 0.8493150684931506,
 'true_positives': 99,
 'false_positives': 28,
 'false_negatives': 5,
 'true_negatives': 87,
 'threshold_used': 0.0,
 'total_correspondences': 5237,
 'filtered_correspondences': 5237,
 'evaluation_timestamp': '2025-12-10T18:13:01.044541',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/companies/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/companies/debug_results_entity_matching/matching_detailed_results.csv']}

In [ ]:
print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

Analyzing cluster size distribution in our entity matching results...


[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 610 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	374	|	61.31%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	98	|	16.07%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	49	|	8.03%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	24	|	3.93%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	10	|	1.64%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	14	|	2.30%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	8	|	1.31%
[INFO ] PyDI.entitymatching.evaluation - 		9	|	6	|	0.98%
[INFO ] PyDI.entitymatching.evaluation - 		10	|	5	|	0.82%
[INFO ] PyDI.entitymatching.evaluation - 		11	|	2	|	0.33%
[INFO ] PyDI.entitymatching.evaluation - 		12	|	4	|	0.66%
[INFO ] PyDI.entitymatching.evaluation - 		13	|	2	|	0.33%
[INFO ] PyDI.entitymatching.evaluation - 		14	


📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,374,61.311475
1,3,98,16.065574
2,4,49,8.032787
3,5,24,3.934426
4,6,10,1.639344
5,7,14,2.295082
6,8,8,1.311475
7,9,6,0.983607
8,10,5,0.819672
9,11,2,0.327869


In [ ]:
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_f2d,
    out_path=cluster_details_path
)

[INFO ] root - Cluster details written to /Users/luca/PycharmProjects/PyDI/usecases/output/companies/cluster_analysis/detailed_cluster_info.json
[INFO ] root - Exported 610 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

In [ ]:
from PyDI.entitymatching import GreedyOneToOneMatchingAlgorithm

# use Greedy One-To-One Matching to refine results to 1:1 matches
clusterer = GreedyOneToOneMatchingAlgorithm()
correspondences_f2d = clusterer.cluster(correspondences_f2d)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2d,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

display(eval_results)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2d,
    out_dir=str(OUTPUT_DIR / "cluster_analysis")
)

print(f"\n📊 Cluster Size Distribution Results:")
display(cluster_distribution)

[INFO ] root - Filtered correspondences: 5237 -> 5237 (threshold=0.0)
[INFO ] root - Greedy matching: 5237 -> 829 correspondences (1658 entities matched)
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 5237 -> 829 correspondences
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 2228 -> 1658 entities
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  91
[INFO ] root -   True Negatives:  110
[INFO ] root -   False Positives: 5
[INFO ] root -   False Negatives: 13
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.918
[INFO ] root -   Precision: 0.948
[INFO ] root -   Recall:    0.875
[INFO ] root -   F1-Score:  0.910


{'precision': 0.9479166666666666,
 'recall': 0.875,
 'f1': 0.91,
 'accuracy': 0.9178082191780822,
 'true_positives': 91,
 'false_positives': 5,
 'false_negatives': 13,
 'true_negatives': 110,
 'threshold_used': 0.0,
 'total_correspondences': 829,
 'filtered_correspondences': 829,
 'evaluation_timestamp': '2025-12-10T18:13:02.362289',
 'output_files': ['/Users/luca/PycharmProjects/PyDI/usecases/output/companies/debug_results_entity_matching/matching_evaluation_summary.json',
  '/Users/luca/PycharmProjects/PyDI/usecases/output/companies/debug_results_entity_matching/matching_detailed_results.csv']}

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 829 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	829	|	100.00%
[INFO ] root - Cluster size distribution written to /Users/luca/PycharmProjects/PyDI/usecases/output/companies/cluster_analysis/cluster_size_distribution.csv



📊 Cluster Size Distribution Results:


,cluster_size,frequency,percentage
0,2,829,100.0


In [ ]:
from PyDI.entitymatching import  MaximumBipartiteMatching

gt_val = load_csv(
    INPUT_DIR / "entitymatching" / "forbes_2_fullcontact_val.csv", 
    name="test_entity_matching",
    header=None,
    names=['id1', 'id2', 'label'],
    add_index=False
)

debug_output_dir = OUTPUT_DIR / "debug_results_entity_matching"
debug_output_dir.mkdir(parents=True, exist_ok=True)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2fc,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2fc,
)

clusterer = GreedyOneToOneMatchingAlgorithm()
correspondences_f2fc = clusterer.cluster(correspondences_f2fc)


cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_f2fc,
)

eval_results = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_f2fc,
    test_pairs=gt_val,
    out_dir=debug_output_dir
)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  211
[INFO ] root -   True Negatives:  455
[INFO ] root -   False Positives: 43
[INFO ] root -   False Negatives: 20
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.914
[INFO ] root -   Precision: 0.831
[INFO ] root -   Recall:    0.913
[INFO ] root -   F1-Score:  0.870
[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 761 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	652	|	85.68%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	63	|	8.28%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	13	|	1.71%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	11	|	1.45%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	3	|	0.39%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	3	|	0.39%
[INFO ] PyDI.entitymatching.evaluation -

Now, we can directly use the trained model with PyDIs MLBasedMatcher

## Part 3: Data Fusion

In [ ]:
forbes["forbes_id"] = forbes["id"]

# Assign trust scores to datasets
forbes.attrs["trust_score"] = 1
dbpedia.attrs["trust_score"] = 3
fullcontact.attrs["trust_score"] = 2

all_correspondences = pd.concat([correspondences_f2d, correspondences_f2fc], ignore_index=True)
print(f'Total correspondences: {len(all_correspondences):,}')

Total correspondences: 1,643


## Step 1: Define Fusion Strategy

In [ ]:
from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, union, prefer_higher_trust, voting, maximum, most_recent

strategy = DataFusionStrategy('company_fusion_strategy')

strategy.add_attribute_fuser('name', voting)
strategy.add_attribute_fuser('assets', prefer_higher_trust)
strategy.add_attribute_fuser('revenue', prefer_higher_trust)
strategy.add_attribute_fuser('keypeople_name', union)
strategy.add_attribute_fuser('founded', prefer_higher_trust)
strategy.add_attribute_fuser('country', voting)
strategy.add_attribute_fuser('city', shortest_string)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'name' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'assets' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'revenue' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'keypeople_name' using rule 'union'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'founded' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'country' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'city' using rule 'shortest_string'


## Step 2: Run Fusion

In [ ]:
from PyDI.fusion import DataFusionEngine

engine = DataFusionEngine(strategy, debug=True, debug_format='json',debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion.jsonl")

fused = engine.run(
    datasets=[forbes, dbpedia, fullcontact],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False,
)
print(f'Fused rows: {len(fused):,}')
display(fused.head(5))

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/companies/data_fusion/debug_fusion.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'company_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 2942 of 2942 unique IDs
[INFO ] PyDI.fusion.engine - Created 12373 record groups from 1643 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 12373 clusters:
[INFO ] PyDI.fusion.engine - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.fusion.engine - 	──────────────────────────────────────────────────
[INFO ] PyDI.fusion.engine - 		2	|	955	|	7.72%
[INFO ] PyDI.fusion.engine - 		3	|	344	|	2.78%
[INFO ] PyDI.fusion.engine - Attribute Consistencies:
[INFO ] PyDI.fusion.engine -     _id: 0.00
[INFO ] PyDI.fusion.engine -     assets: 0.86
[INFO ] PyDI.fusion.engine -     city:

Fused rows: 1,299


,_id,_fusion_sources,_fusion_source_datasets,website,country,id,founded,revenue,industry,name_first_token,forbes_id,assets,keypeople_name,city,name,_fusion_confidence,_fusion_metadata
0,http://www.forbes.com/companies/bilfinger/,"[http://www.forbes.com/companies/bilfinger/, f...","[forbes, fullcontact]",http://www.forbes.com/companies/bilfinger/,Germany,http://www.forbes.com/companies/bilfinger/,1880-01-01,11200000000,Construction Services,bilfinger,http://www.forbes.com/companies/bilfinger/,9000000000,None,Mannheim,Bilfinger,0.727273,"{'website_rule': 'first_non_null', 'website_in..."
1,http://dbpedia.org/resource/Devon_Motorworks,"[http://dbpedia.org/resource/Devon_Motorworks,...","[dbpedia, forbes]",http://www.forbes.com/companies/devon-energy/,United States,http://dbpedia.org/resource/Devon_Motorworks,2008-01-01,10800000000,Industrial design,devon,http://www.forbes.com/companies/devon-energy/,42900000000,None,Los Angeles,Devon Motorworks,0.636364,"{'website_rule': 'first_non_null', 'website_in..."
2,fullcontact_1316,"[fullcontact_1316, http://www.forbes.com/compa...","[fullcontact, forbes, dbpedia]",http://www.forbes.com/companies/chesapeake-ene...,United States,fullcontact_1316,1967-01-01,17800000000,Oil & Gas Operations,chesapeake,http://www.forbes.com/companies/chesapeake-ene...,41800000000,None,Annapolis,Chesapeake Bay Foundation,0.651515,"{'website_rule': 'first_non_null', 'website_in..."
3,http://dbpedia.org/resource/Aareal_Bank,"[http://dbpedia.org/resource/Aareal_Bank, http...","[dbpedia, forbes, fullcontact]",http://www.forbes.com/companies/aareal-bank/,Germany,http://dbpedia.org/resource/Aareal_Bank,1923-01-01,1300000000,Financial services,aareal,http://www.forbes.com/companies/aareal-bank/,41220000000,None,Wiesbaden,Aareal Bank,0.651515,"{'website_rule': 'first_non_null', 'website_in..."
4,http://dbpedia.org/resource/Prudential_plc,"[http://dbpedia.org/resource/Prudential_plc, h...","[dbpedia, forbes, fullcontact]",http://www.forbes.com/companies/prudential/,United Kingdom,http://dbpedia.org/resource/Prudential_plc,1848-01-01,30502000000,Financial services,prudential,http://www.forbes.com/companies/prudential/,528500000000,None,London,Prudential plc,0.590909,"{'website_rule': 'first_non_null', 'website_in..."


## Step 3: Evaluate Data Fusion

In [ ]:
from PyDI.fusion import tokenized_match, year_only_match, set_equality_match, numeric_tolerance_match

strategy.add_evaluation_function("name", tokenized_match)
strategy.add_evaluation_function("assets", tokenized_match)
strategy.add_evaluation_function("revenue", numeric_tolerance_match, tolerance=0.1)
strategy.add_evaluation_function("assets", numeric_tolerance_match, tolerance=0.1)
strategy.add_evaluation_function("keypeople_name", set_equality_match)
strategy.add_evaluation_function("founded", year_only_match)
strategy.add_evaluation_function("country", tokenized_match)
strategy.add_evaluation_function("city", tokenized_match)

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'assets'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'revenue' with params {'tolerance': 0.1}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'assets' with params {'tolerance': 0.1}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'keypeople_name'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'founded'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'country'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'city'


In [ ]:
from PyDI.fusion import DataFusionEvaluator

fusion_test_set = load_xml(INPUT_DIR / 'fusion' / 'test_set.xml', name='fusion_test_set', nested_handling='aggregate')
fusion_test_set['keypeople_name'] = fusion_test_set['keypeople_name'].apply(lambda x: [x] if isinstance(x, str) else x)

# Create evaluator with our fusion strategy
evaluator = DataFusionEvaluator(strategy, debug=True, debug_file=OUTPUT_DIR / "data_fusion" / "debug_fusion_eval.jsonl", debug_format="json")

# Evaluate the fused results against the gold standard
print("Evaluating fusion results against gold standard...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,
    fused_id_column='forbes_id',
    gold_df=fusion_test_set,
    gold_id_column='id',
)

# Display evaluation metrics
print("\nFusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")

[INFO ] PyDI.fusion.evaluation - Fusion evaluation debug logging enabled; refer to /Users/luca/PycharmProjects/PyDI/usecases/output/companies/data_fusion/debug_fusion_eval.jsonl for mismatch details.
[INFO ] PyDI.fusion.evaluation - Starting fusion evaluation
[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.872 overall accuracy (102/117)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 15 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	revenue                          |       3 |     20.00%%
[INFO ] PyDI.fusion.evaluation - 	keypeople_name                   |       3 |     20.00%%
[INFO ] PyDI.fusion.evaluation - 	city                             |       3 |     20.00%%
[INFO ] PyDI.fusion.evaluation - 	name                             |       3 |     20.00%%
[INFO ] PyDI.fusion.eva

Evaluating fusion results against gold standard...

Fusion Evaluation Results:
  overall_accuracy: 0.872
  macro_accuracy: 0.857
  num_evaluated_records: 18
  num_evaluated_attributes: 7
  total_evaluations: 117
  total_correct: 102
  country_accuracy: 0.944
  country_count: 18
  founded_accuracy: 0.889
  founded_count: 18
  revenue_accuracy: 0.833
  revenue_count: 18
  assets_accuracy: 1.000
  assets_count: 18
  keypeople_name_accuracy: 0.667
  keypeople_name_count: 9
  city_accuracy: 0.833
  city_count: 18
  name_accuracy: 0.833
  name_count: 18

Overall Accuracy: 87.2%
